## Evaluate Static Profitability Scenarios

**Purpose**
Runs static profitability and subsidy diagnostics under multiple assumptions to build comparative outputs.

**Primary inputs**
- `project/input technical/economic files loaded through get_inputs/get_config`

**Primary outputs**
- `runs/static_analysis_output/*.csv`
- `runs/static_analysis_output/*.png`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Static Analysis (Comprehensive)

**Purpose:** Full profitability and subsidy analysis across scenarios — cost-benefit, distributional effects.

**Project imports:** `project.model`, `project.utils`, `project.input.resources`.

**Inputs:** Building stock, energy prices, costs, subsidies.

**Outputs:** Profitability tables, subsidy disaggregation, scenario comparisons.


In [ ]:
import os
import pandas as pd
import numpy as np
from project.model import get_inputs
from project.utils import make_stackedbar_plot
from project.utils import make_plots, reindex_mi



### Load inputs


In [ ]:
input_resirf = get_inputs()
year = 2018
buildings = input_resirf['buildings']
prices = input_resirf['energy_prices'].loc[year, :]
carbon_emission = input_resirf['carbon_emission'].loc[year, :]
cost_insulation = input_resirf['cost_insulation']
cost_heater = input_resirf['cost_heater']
health_cost_dpe = input_resirf['health_cost_dpe']
present_discount_rate = input_resirf['present_discount_rate']
income = input_resirf['income']
interest_rate = 0.039
duration_loan = 10

cost_financing = cost_insulation * interest_rate * duration_loan
cost_insulation = cost_insulation + cost_financing
cost_financing = cost_heater * interest_rate * duration_loan
cost_heater = cost_heater + cost_financing

carbon_value = 250
social_discount_rate = 0.032
lifetime = 25

cost_failures_landlords = pd.Series([0, 18000, 18000], index=pd.Index(['Owner-occupied', 'Privately rented', 'Social-housing'], name='Occupancy status'))
cost_failures_multi = pd.Series([0, 14000], index=pd.Index(['Single-family', 'Multi-family'], name='Housing type'))
# Creating the double index
index = pd.MultiIndex.from_product([cost_failures_landlords.index, cost_failures_multi.index], names=['Occupancy status', 'Housing type'])

# Initialize the combined series with the double index
cost_failures = pd.Series(index=index, dtype=int)

# Populate the combined series with sums
for idx in cost_failures.index:
    cost_failures[idx] = cost_failures_landlords[idx[0]] + cost_failures_multi[idx[1]]


### Load scenarios
#### Profitability of measures for different scenarios


In [ ]:
selected_options = [
    "I-E-C-S",
    "I-E-C",
    "I-E-S",
    "I-E social discount",
    "I-E",
    "I-E-MF Bailleurs",
    "I-E-MF Bailleurs + Collectifs",
    "I-E-MF Contrainte crédit"
]

sufix = '_isolation_chauffage'

rslt_profitability, _ = buildings.make_static_analysis(cost_insulation, cost_heater,
                                                       prices, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=social_discount_rate,
                                                       private_discount_rate=present_discount_rate,
                                                       carbon_value=carbon_value, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, sufix=sufix,
                                                       lifetime=lifetime, make_plot=True,
                                                       cost_failures=cost_failures,
                                                       cost_failures_landlords=cost_failures_landlords,
                                                       income=income, duration_loan=duration_loan)



#### Compare technologies


In [ ]:
selected_options = [
    "I-E-C-S, isolation & chauffage",
    "I-E-C-S, isolation"
]
sufix = '_comparison_technologies'
rslt_profitability, _ = buildings.make_static_analysis(cost_insulation, cost_heater,
                                                       prices, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=social_discount_rate,
                                                       private_discount_rate=present_discount_rate,
                                                       carbon_value=carbon_value, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, sufix=sufix,
                                                       lifetime=lifetime, make_plot=True,
                                                       cost_failures=cost_failures)


#### Only insulation


### Sensitivity analysis
Share of emission, consumption and dwellings that are profitable. Based on NPV.


In [ ]:

carbon_values = [100, 250, 500]
discount_rates = [0.02, 0.032, 0.05]
factors_cost = [0.5, 1.5]
factors_price = [0.5, 1.5, 2]


selected_options = [
    "I-E-C-S",
    "I-E",
    "I-E-MF Contrainte crédit"
]

name = 'Carbon value'
rslt_sensitivity, k = {}, 0
for cv in carbon_values:
    print(cv)
    dr, fc, fp = social_discount_rate, 1, 1
    rslt_profitability, _ = buildings.make_static_analysis(cost_insulation * fc, cost_heater,
                                                       prices * fp, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=dr, private_discount_rate=present_discount_rate,
                                                       carbon_value=cv, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, lifetime=lifetime,
                                                       make_plot=False, cost_failures=cost_failures,
                                                       cost_failures_landlords=cost_failures_landlords,
                                                       income=income)
    
    rslt_sensitivity.update({(name, cv): pd.DataFrame(rslt_profitability).T})
print('Carbon value done')

name = 'Social discount rate'
for dr in discount_rates:
    print(dr)
    cv, fc, fp = carbon_value, 1, 1
    rslt_profitability, _ = buildings.make_static_analysis(cost_insulation * fc, cost_heater,
                                                       prices * fp, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=dr, private_discount_rate=present_discount_rate,
                                                       carbon_value=cv, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, lifetime=lifetime,
                                                       make_plot=False, cost_failures=cost_failures,
                                                       cost_failures_landlords=cost_failures_landlords,
                                                       income=income)
    rslt_sensitivity.update({(name, dr): pd.DataFrame(rslt_profitability).T})
print('Discount rate done')
  
name = 'Factor cost'
for fc in factors_cost:
    print(fc)

    cv, dr, fp = social_discount_rate, social_discount_rate, 1
    rslt_profitability, _ = buildings.make_static_analysis(cost_insulation * fc, cost_heater,
                                                       prices * fp, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=dr, private_discount_rate=present_discount_rate,
                                                       carbon_value=cv, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, lifetime=lifetime,
                                                       make_plot=False, cost_failures=cost_failures,
                                                       cost_failures_landlords=cost_failures_landlords,
                                                       income=income)
    rslt_sensitivity.update({(name, fc): pd.DataFrame(rslt_profitability).T})
print('Factor cost done')

name = 'Factor price'
for fp in factors_price:
    print(fp)

    cv, dr, fc = social_discount_rate, social_discount_rate, 1
    rslt_profitability, _ = buildings.make_static_analysis(cost_insulation * fc, cost_heater,
                                                       prices * fp, health_cost_dpe, carbon_emission,
                                                       social_discount_rate=dr, private_discount_rate=present_discount_rate,
                                                       carbon_value=cv, path_out=os.path.join('runs', 'static_analysis_output'),
                                                       selected_options=selected_options, lifetime=lifetime,
                                                       make_plot=False, cost_failures=cost_failures,
                                                       cost_failures_landlords=cost_failures_landlords,
                                                       income=income)
    rslt_sensitivity.update({(name, fp): pd.DataFrame(rslt_profitability).T})
print('Factor price done')

cv, dr, fc, fp = carbon_value, social_discount_rate, 1, 1
rslt_profitability, _ = buildings.make_static_analysis(cost_insulation * fc, cost_heater,
                                                   prices * fp, health_cost_dpe, carbon_emission,
                                                   social_discount_rate=dr, private_discount_rate=present_discount_rate,
                                                   carbon_value=cv, path_out=os.path.join('runs', 'static_analysis_output'),
                                                   selected_options=selected_options, lifetime=lifetime,
                                                   make_plot=False, cost_failures=cost_failures,
                                                   cost_failures_landlords=cost_failures_landlords,
                                                   income=income)
rslt_sensitivity.update({('Reference', 'Reference'): pd.DataFrame(rslt_profitability).T})


rslt_sensitivity = pd.concat(rslt_sensitivity)
rslt_sensitivity.index.names = ['Variable', 'Value', 'Output']
rslt_sensitivity = rslt_sensitivity.reorder_levels(['Output', 'Variable', 'Value'])
rslt_sensitivity = rslt_sensitivity.sort_index(level='Output')
rslt_sensitivity.to_csv(os.path.join('runs', 'static_analysis_output', 'sensitivity_analysis_oat.csv'), encoding='utf-8-sig')


#### Sensitivity analysis with combination of all inputs


### Subsidies estimation


In [ ]:
selected_options = [
    "I-E-C-S, isolation & chauffage"
]
sufix = '_isolation_chauffage'
rslt_profitability, dict_subsidies = buildings.make_static_analysis(cost_insulation, cost_heater,
                                                                    prices, health_cost_dpe, carbon_emission,
                                                                    social_discount_rate=social_discount_rate,
                                                                    private_discount_rate=present_discount_rate,
                                                                    carbon_value=carbon_value, path_out=os.path.join('runs', 'static_analysis_output'),
                                                                    selected_options=selected_options, sufix=sufix,
                                                                    lifetime=lifetime, make_plot=False, 
                                                                    cost_failures=cost_failures)
subsidies_details = dict_subsidies["I-E-C-S, isolation & chauffage"].sort_values('criteria', ascending=False)
subsidies_details['Revenu (€/an.logement)'] = reindex_mi(income.rename_axis(['Income owner']), subsidies_details.index)
subsidies_details['Ratio coût sur revenu'] = subsidies_details["Coût d'investissement (€/logement)"] / duration_loan / subsidies_details['Revenu (€/an.logement)']
subsidies_details["Subvention contrainte de crédit (€/logement)"] = (subsidies_details['Ratio coût sur revenu'] > 0.05) * (subsidies_details["Coût d'investissement (€/logement)"] - 0.05 * duration_loan * subsidies_details['Revenu (€/an.logement)'])
subsidies_details['Present discount rate (%)'] = reindex_mi(present_discount_rate, subsidies_details.index)
subsidies_details['Present discount factor'] = (1 - (1 + subsidies_details['Present discount rate (%)'] ) ** -lifetime) / subsidies_details['Present discount rate (%)'] 
subsidies_details['Coût privé I-E (€/logement)'] = subsidies_details["Coût d'investissement (€/logement)"] - subsidies_details['Present discount factor'] * subsidies_details["Économie de facture (€/(an.logement))"]
subsidies_details['Coût privé I-E-MF (€/logement)'] = subsidies_details['Coût privé I-E (€/logement)'] + subsidies_details["Coût des frictions en forme réduite (€/logement)"]
subsidies_details["Subvention (I-E) (€/logement)"] = (subsidies_details["Coût actualisé social (€/logement)"] < 0) * (subsidies_details["Coût privé I-E (€/logement)"] > 0) * subsidies_details["Coût privé I-E (€/logement)"]
subsidies_details["Subvention (I-E) (€)"] = subsidies_details["Subvention (I-E) (€/logement)"] * subsidies_details["Stock de résidences principales (unité)"]
subsidies_details["Subvention (I-E-MF) (€/logement)"] = (subsidies_details["Coût actualisé social (€/logement)"] < 0) * (subsidies_details["Coût privé I-E-MF (€/logement)"] > 0) * subsidies_details["Coût privé I-E-MF (€/logement)"]
subsidies_details["Subvention (I-E-MF Contrainte crédit) (€/logement)"] = subsidies_details[["Subvention (I-E-MF) (€/logement)", "Subvention contrainte de crédit (€/logement)"]].max(axis=1)
subsidies_details["Subvention (I-E-MF Contrainte crédit) (€)"] = subsidies_details["Subvention (I-E-MF Contrainte crédit) (€/logement)"] * subsidies_details["Stock de résidences principales (unité)"]

subsidies_details.to_csv(os.path.join('runs', 'static_analysis_output', 'details_calculation_subsidies.csv'), encoding='utf-8-sig')


In [ ]:
emission_total = 49792611347424.03
x_axis = "Économie d'émission (gCO2/an)"
temp = subsidies_details.loc[:, [x_axis, 'Coût privé I-E (€/logement)', "Coût actualisé social (€/logement)", 'Measures']].sort_values("Coût actualisé social (€/logement)", ascending=True)
dict_df = {}
df = temp.loc[:, [x_axis, 'Coût privé I-E (€/logement)']]
df[x_axis] = df[x_axis].cumsum() / emission_total
dict_df.update({'Private': df.set_index(x_axis).squeeze().copy()})
df = temp.loc[:, [x_axis, "Coût actualisé social (€/logement)"]]
df[x_axis] = df[x_axis].cumsum() / emission_total
dict_df.update({'Social': df.set_index(x_axis).squeeze().copy()})
make_plots(dict_df, 'Coût actualisé (Millers €)', integer=False, save=os.path.join('runs', 'static_analysis_output', 'comparison_cost_private_social.png'), ymin=-40000, format_y=lambda y, _: '{:.0f}'.format(y/1e3), format_x=lambda x, _: '{:.0%}'.format(x))


#### Description of household to target with subsidies

Calculate subsidies as the difference between the net present cost and 0 


In [ ]:
subsidies_details.columns


In [ ]:
def weighted_average(group, avg_name='value', weight_name='Stock de résidences principales (unité)'):
    return (group[avg_name] * group[weight_name]).sum() / group[weight_name].sum()

def describe_categories(df):
    
    columns_mean = ["Économie d'énergie (kWh/an.logement)", "Économie d'émission (gCO2/an.logement)", "Coût d'investissement (€/logement)", 'Subvention (I-E) (€/logement)', 'Subvention (I-E-MF Contrainte crédit) (€/logement)']
    columns_sum = ['Stock de résidences principales (unité)']
    df_category = pd.DataFrame()
    
    if 'measures' in df.columns:
        df.set_index('measures', inplace=True, append=True)
        
    group = df.index.get_level_values('Energy').isin(['Natural gas', 'Oil fuel']) & df.index.get_level_values('Performance').isin(['F', 'G']) & df.index.get_level_values('Income tenant').isin(['C1', 'C2'])
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby(group).apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: 'Groupe cible {}'.format(x))
        temp.index.names = ['Groupe cible']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby(group)[columns_sum].sum()
    temp.index = temp.index.map(lambda x: 'Groupe cible {}'.format(x))
    temp.index.names = ['Groupe cible']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
        
                     
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Income tenant').apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: 'Revenu occupant {}'.format(x))
        temp.index.names = ['Income tenant']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby('Income tenant')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: 'Revenu occupant {}'.format(x))
    temp.index.names = ['Income tenant']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Income owner').apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: 'Revenu propriétaire {}'.format(x))
        temp.index.names = ['Income owner']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby('Income owner')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: 'Revenu propriétaire {}'.format(x))
    temp.index.names = ['Income owner']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Occupancy status').apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: '{}'.format(x))
        temp.index.names = ['Occupancy status']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby('Occupancy status')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: '{}'.format(x))
    temp.index.names = ['Occupancy status']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Housing type').apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: '{}'.format(x))
        temp.index.names = ['Housing type']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby('Housing type')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: '{}'.format(x))
    temp.index.names = ['Housing type']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Performance').apply(weighted_average, avg_name=c)
        temp.index = temp.index.map(lambda x: 'DPE {}'.format(x))
        temp.index.names = ['Performance']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
    
    temp = df.groupby('Performance')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: 'DPE {}'.format(x))
    temp.index.names = ['Performance']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    df_group = pd.DataFrame()
    for c in columns_mean:
        temp = df.groupby('Energy').apply(weighted_average, avg_name=c)
        temp.index.names = ['Energy']
        temp.name = c
        df_group = pd.concat((df_group, temp), axis=1)
        
    temp = df.groupby('Energy')[columns_sum].sum()
    temp.index = temp.index.map(lambda x: '{}'.format(x))
    temp.index.names = ['Energy']
    temp.name = c
    temp /= temp.sum()
    df_group = pd.concat((temp, df_group), axis=1)
    df_category = df_category.append(df_group)
    
    if 'measures' in df.index.names:
        df_group = pd.DataFrame()
        for c in columns_mean:
            temp = df.groupby('measures').apply(weighted_average, avg_name=c)
            temp.index = temp.index.map(lambda x: '{}'.format(x))
            temp.index.names = ['measures']
            temp.name = c
            df_group = pd.concat((df_group, temp), axis=1)
            
        temp = df.groupby('measures')[columns_sum].sum()
        temp.index = temp.index.map(lambda x: '{}'.format(x))
        temp.index.names = ['measures']
        temp.name = c
        temp /= temp.sum()
        df_group = pd.concat((temp, df_group), axis=1)
        df_category = df_category.append(df_group)
    
    df_category["Économie d'énergie (kWh/an.logement)"] /= 1e3
    df_category["Économie d'émission (gCO2/an.logement)"] /= 1e6
    if 'Subvention (I-E) (€/logement)' in df_category.columns:
        df_category['Subventions moyenne (I-E) (%)'] = df_category['Subvention (I-E) (€/logement)'] / df_category["Coût d'investissement (€/logement)"]
    
    if 'Subvention (I-E-MF Contrainte crédit) (€/logement)' in df_category.columns:
        df_category['Subventions moyenne (I-E-MF Contrainte crédit) (%)'] = df_category['Subvention (I-E-MF Contrainte crédit) (€/logement)'] / df_category["Coût d'investissement (€/logement)"]
    
    return df_category
    


In [ ]:
x_indicator = "Stock de résidences principales (unité)"
column_subsidies = "Subvention (I-E-MF Contrainte crédit) (€/logement)"
df = subsidies_details.copy()
df = buildings.add_certificate(df)
df = buildings.add_energy(df)
df = df.loc[df[column_subsidies] > 0, :]
df.sort_values("Coût actualisé social (€/logement)", ascending=True, inplace=True)
df['{}_cumulated'.format(x_indicator)] = df[x_indicator].cumsum() / df[x_indicator].sum()
rslt_range = {}
for i in range(0, 100, 10):
    print(i)
    df_range = df.loc[(df['{}_cumulated'.format(x_indicator)] > i / 100) & (df['{}_cumulated'.format(x_indicator)] < (i + 10) / 100), :]
    df_temp = describe_categories(df_range)
    rslt_range.update({'{}%-{}%'.format(i, i+10): df_temp})
    
rslt_range = pd.concat(rslt_range, axis=0)
rslt_range.index.names = ['Range', 'Category']
rslt_range.to_csv(os.path.join('runs', 'static_analysis_output', 'subsidies_grouped_range.csv'), encoding='utf-8-sig')


In [ ]:
# Make stacked bar plot for each category

from project.input.resources import resources_data
figs = {'Revenu propriétaire': {'Revenu propriétaire {}'.format(i): i for i in ['C1', 'C2', 'C3', 'C4', 'C5']},
        'Revenu occupant': {'Revenu occupant {}'.format(i): i for i in ['C1', 'C2', 'C3', 'C4', 'C5']},
        "Statut d'occupation": {'{}'.format(i): i for i in ['Owner-occupied', 'Privately rented', 'Social-housing']},
        "Type de logement": {'{}'.format(i): i for i in ['Single-family', 'Multi-family']},
        "Performance": {'DPE {}'.format(i): i for i in ['A', 'B', 'C', 'D', 'E', 'F', 'G']},
        "Énergie": {'{}'.format(i): i for i in ['Natural gas', 'Oil fuel', "Electricity", "Heating", "Wood fuel"]},
        }


for key, item in figs.items():

    temp = rslt_range.loc[rslt_range.index.get_level_values('Category').isin(item.keys()), :]
    temp = temp.loc[:, 'Stock de résidences principales (unité)']
    temp = temp.reset_index()
    temp['Category'] = temp['Category'].map(lambda x: item[x])
    temp.set_index(['Category', 'Range'], inplace=True)
    temp = temp.squeeze().unstack('Category')
    temp.to_csv(os.path.join('runs', 'static_analysis_output', 'subsidies_grouped_range_{}.csv').format(key))
    make_stackedbar_plot(temp, '{} (Nombre de logements cibles)'.format(key), format_y=lambda y, _: '{:.0%}'.format(y), rotation=90, colors=resources_data['colors'], save=os.path.join('runs', 'static_analysis_output', 'subsidies_grouped_range_{}.png').format(key),
                         left=1.2)



In [ ]:
df['Stock de résidences principales (unité)'].sum() / 1e6
